In [1]:
import argparse
import json
from langchain.document_loaders import PyPDFLoader
from langchain import PromptTemplate, LLMChain
from langchain.llms.ollama import Ollama
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

In [2]:

# Define the schema for each skill and the overall output.
class Skill(BaseModel):
    skill: str
    influence: int = Field(..., ge=0, le=100)  # Influence percentage (0-100)

class ResumeSkills(BaseModel):
    soft_skills: List[Skill]
    hard_skills: List[Skill]


In [3]:

# Create an output parser using the defined Pydantic model.
output_parser = PydanticOutputParser(pydantic_object=ResumeSkills)
# Get the format instructions (JSON schema) from the parser.
format_instructions = output_parser.get_format_instructions()
# Escape curly braces so they are not interpreted as template variables.
escaped_format_instructions = format_instructions.replace("{", "{{").replace("}", "}}")


In [4]:
# Create the prompt template with the escaped format instructions.
prompt_template = f"""
You are given resume text extracted from a PDF file. Your task is to extract the soft skills and hard skills mentioned in the resume.
Soft skills include communication, teamwork, adaptability, problem-solving, leadership, emotional intelligence, and time management.
Hard skills include the following:
- Programming and Software Development
- Data Analysis and Statistical Analysis
- Project Management
- Financial Analysis and Forecasting
- Technical Writing and Documentation
- Machine Learning and Artificial Intelligence
- Graphic Design and Visual Communication
- Digital Marketing and SEO/SEM
- Web Development
- Database Management and SQL
- Cybersecurity and Information Security
- IT Networking and Infrastructure Management
- Quality Assurance and Software Testing
- Computer-Aided Design (CAD) and 3D Modeling
- Engineering Design and Simulation
- Scientific Research and Laboratory Skills
- Legal Research and Compliance
- Social Media Management and Analytics
- Content Creation and Copywriting
- Multimedia Production and Video Editing
- Technical Support and Troubleshooting
- Operating Systems Administration
- DevOps and Continuous Integration/Deployment
- Agile and Scrum Methodologies
- Data Visualization
- Business Intelligence and Analytics
- Supply Chain Management and Logistics
- Sales and Negotiation Techniques
- Advanced Excel and Data Modeling
- Statistical Software Proficiency (R, SAS, SPSS)
- Cloud Computing (AWS, Azure, Google Cloud)
- Mobile Application Development
- Robotics and Automation Engineering
- Virtual Reality (VR) and Augmented Reality (AR) Development
- E-commerce Platform Management
- Digital Forensics and Incident Response
- Network Security Monitoring and Penetration Testing
- Biotechnology Techniques and Laboratory Procedures
- Geographic Information Systems (GIS) and Spatial Analysis
- Foreign Language Proficiency
- Medical Diagnosis and Patient Care
- Mechanical Engineering Design and Analysis
- Electronics Engineering and Circuit Design
- Management Consulting and Strategic Advisory

For each skill you extract, assign a percentage influence (from 0 to 100) that reflects how prominently the skill is represented in the resume.
Return the output in JSON format following this schema:
{escaped_format_instructions}

Resume:
{{resume_text}}
"""


In [5]:
# Create the PromptTemplate; note that only "resume_text" is the input variable.
template = PromptTemplate(
    input_variables=["resume_text"],
    template=prompt_template,
)


In [6]:

# Initialize the Ollama LLM with the model running on localhost.
llm = Ollama(model="llama3:8b", temperature=0, base_url="http://localhost:11434")

C:\Users\Subhayan Das\AppData\Local\Temp\ipykernel_31172\3825228862.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3:8b", temperature=0, base_url="http://localhost:11434")


In [7]:

# Create an LLMChain with the prompt and output parser.
chain = LLMChain(llm=llm, prompt=template, output_parser=output_parser)


C:\Users\Subhayan Das\AppData\Local\Temp\ipykernel_31172\703957443.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=template, output_parser=output_parser)


In [8]:
def extract_text_from_pdf(pdf_file_path: str) -> str:
    """
    Uses LangChain's PyPDFLoader to load and extract text from a PDF resume.
    """
    loader = PyPDFLoader(pdf_file_path)
    docs = loader.load()  # load() returns a list of Document objects.
    # Combine text from all pages.
    text = "\n".join(doc.page_content for doc in docs)
    return text

def main(resume_file_path: str, output_json_path: str):
    # Extract text from the PDF resume.
    resume_text = extract_text_from_pdf(resume_file_path)
    
    # Run the LLM chain to extract skills and their influence percentages.
    result = chain.run(resume_text=resume_text)
    
    # Dump the result (a Pydantic model) to a dict and then convert to pretty JSON.
    result_dict = result.model_dump()
    json_result = json.dumps(result_dict, indent=2)
    print(json_result)
    
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(result_dict, f, indent=2)
    
    print(f"✅ Skills extracted and saved to: {output_json_path}")

In [10]:
if __name__ == "__main__":
    file_path = "C:/Users/Subhayan Das/Desktop/Course/Adaptive Applications/Project/User Modeling/data/SDE Resume.pdf"
    output_path = "C:/Users/Subhayan Das/Desktop/Course/Adaptive Applications/Project/User Modeling/output/Subhayan1.json"
    main(file_path, output_path)


C:\Users\Subhayan Das\AppData\Local\Temp\ipykernel_31172\3479184591.py:16: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = chain.run(resume_text=resume_text)


{
  "soft_skills": [
    {
      "skill": "Communication",
      "influence": 80
    },
    {
      "skill": "Teamwork",
      "influence": 90
    },
    {
      "skill": "Adaptability",
      "influence": 70
    }
  ],
  "hard_skills": [
    {
      "skill": "Machine Learning",
      "influence": 95
    },
    {
      "skill": "Python",
      "influence": 90
    },
    {
      "skill": "SQL",
      "influence": 85
    },
    {
      "skill": "Data Analysis",
      "influence": 80
    },
    {
      "skill": "Project Management",
      "influence": 75
    }
  ]
}
✅ Skills extracted and saved to: C:/Users/Subhayan Das/Desktop/Course/Adaptive Applications/Project/User Modeling/output/Subhayan1.json
